# 02 Inspect CLEAN

- apre il parquet CLEAN prodotto dal toolkit
- mostra schema, preview e sanity checks minimi
- aiuta a verificare `clean.required_columns` e `clean.validate`

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import duckdb
import yaml

ROOT = Path('.').resolve()
DATASET_YML = (ROOT / 'dataset.yml').resolve() if (ROOT / 'dataset.yml').exists() else (ROOT / '..' / 'dataset.yml').resolve()
CFG = yaml.safe_load(DATASET_YML.read_text(encoding='utf-8'))
DATASET = CFG['dataset']['name']
YEARS = CFG['dataset']['years']
YEAR_INDEX = 0
YEAR = YEARS[YEAR_INDEX] if YEARS and 0 <= YEAR_INDEX < len(YEARS) else YEARS[0]
CLI_PREFIX = ['toolkit'] if shutil.which('toolkit') else ['py', '-m', 'toolkit.cli.app']
INSPECT_CMD = CLI_PREFIX + ['inspect', 'paths', '--config', str(DATASET_YML), '--year', str(YEAR), '--json']
INSPECT = json.loads(subprocess.run(INSPECT_CMD, capture_output=True, text=True, check=True).stdout)
CLEAN_DIR = Path(INSPECT['paths']['clean']['dir'])
CLEAN_PATH = Path(INSPECT['paths']['clean']['output'])
REQUIRED_COLUMNS = CFG.get('clean', {}).get('required_columns', [])
PRIMARY_KEY = CFG.get('clean', {}).get('validate', {}).get('primary_key', [])

{
    'YEARS': YEARS,
    'YEAR_INDEX': YEAR_INDEX,
    'CLEAN_PATH': str(CLEAN_PATH),
    'INSPECT_CMD': INSPECT_CMD,
    'REQUIRED_COLUMNS': REQUIRED_COLUMNS,
    'PRIMARY_KEY': PRIMARY_KEY,
}

In [ ]:
con = duckdb.connect()

if CLEAN_PATH.exists():
    schema_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{CLEAN_PATH.as_posix()}')").df()
    preview_df = con.execute(f"SELECT * FROM read_parquet('{CLEAN_PATH.as_posix()}') LIMIT 20").df()
    display(schema_df)
    display(preview_df)
else:
    print('CLEAN parquet not found. Run toolkit run clean --config dataset.yml first.')

In [ ]:
if CLEAN_PATH.exists():
    row_count = con.execute(f"SELECT COUNT(*) AS row_count FROM read_parquet('{CLEAN_PATH.as_posix()}')").df()
    display(row_count)

    if REQUIRED_COLUMNS:
        available = {row[0] for row in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{CLEAN_PATH.as_posix()}')").fetchall()}
        print({'missing_required_columns': [col for col in REQUIRED_COLUMNS if col not in available]})

    if PRIMARY_KEY:
        keys = ', '.join(PRIMARY_KEY)
        dup_df = con.execute(
            f"SELECT {keys}, COUNT(*) AS dup_count FROM read_parquet('{CLEAN_PATH.as_posix()}') GROUP BY {keys} HAVING COUNT(*) > 1 LIMIT 20"
        ).df()
        display(dup_df)
else:
    print('No CLEAN output available.')